In [17]:
from maq20 import MAQ20

IP = "192.168.1.40"
PORT = 502
MODULE = 6

m = MAQ20(ip_address=IP, port=PORT)
m.scan_module_list()

mod = m.get_module(MODULE)

BASE = MODULE*2+100  

# =========================================================
# WRITE
# =========================================================
def write_channels(channels):
    if len(channels) != 5:
        raise ValueError("Need exactly 10 channels")

    for i, val in enumerate(channels):
        addr = BASE + i
        mod.write_register(addr, int(val))
        print(f"Wrote DO{i} -> addr {addr} = {val}")


# =========================================================
# READ
# =========================================================
def read_channels():
    values = []

    for i in range(5):
        addr = BASE + i
        val = mod.read_register(addr)

        # normalize (maq20 sometimes returns bool)
        val = int(val)

        values.append(val)
        print(f"Read DO{i} <- addr {addr} = {val}")

    return values


# =========================================================
# TEST
# =========================================================
channels = [1,0,1,1,0]

print("\n--- WRITE ---")
write_channels(channels)

print("\n--- READ BACK ---")
read_values = read_channels()

# =========================================================
# CHECK
# =========================================================
print("\nExpected:", channels)
print("Read    :", read_values)

if read_values == channels:
    print("\n✅ MATCH")
else:
    print("\n❌ MISMATCH")


--- WRITE ---
Wrote DO0 -> addr 112 = 1
Wrote DO1 -> addr 113 = 0
Wrote DO2 -> addr 114 = 1
Wrote DO3 -> addr 115 = 1
Wrote DO4 -> addr 116 = 0

--- READ BACK ---
Read DO0 <- addr 112 = 1
Read DO1 <- addr 113 = 0
Read DO2 <- addr 114 = 1
Read DO3 <- addr 115 = 1
Read DO4 <- addr 116 = 0

Expected: [1, 0, 1, 1, 0]
Read    : [1, 0, 1, 1, 0]

✅ MATCH


In [ ]:
from maq20 import MAQ20
from pymodbus.client import ModbusTcpClient

IP = "192.168.1.40"
PORT = 502
SLAVE = 1

# Discover module
maq20 = MAQ20(ip_address=IP, port=PORT)
diol = maq20.find("DIOL")

if diol:
    R = 6
    addr = 2000 * R + 1000   # DO0

    client = ModbusTcpClient(IP, port=PORT)
    client.connect()

    # Turn ON (0 = ON)
    # client.write_register(addr, 0, slave=SLAVE)
    # client.write_register(addr, 1, slave=SLAVE)

    # Read back using maq20
    value = diol.read_data(2)
    state = "ON" if value == 0 else "OFF"

    print(f"DO0 state: {state} (raw={value})")

    client.close()

else:
    print("DIOL module not found.")

DO0 state: OFF (raw=[0])
